# 21 - Analyze two-model selective refinement (v2)

This notebook combines the existing uncertainty-only arms with the matched K=5 `(3,4)`
refine-last arms for both competent v2 models. The primary predeclared policy uses first-chunk
uncertainty: select the lower-U model, refine it for the episode only when U is at least `0.03`.

Prefix, individual-chunk, and full-episode summaries are also analyzed. Only the first-chunk rule
is directly deployable from these independently simulated trajectories. Later scores are useful
post-hoc evidence about predictiveness and headroom, but they are not presented as an online
policy. The best in-sample window is exploratory; leave-one-suite-out (LOSO) window selection is
reported separately to reduce threshold-selection optimism.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch and validate the exact four-arm cohort

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from pnp.config import Method
from pnp.store import SupabaseStore
from pnp.diversity import (DIVERSITY_FIXED_REFINEMENT_THRESHOLD,
    DIVERSITY_V2_EXPERIMENT_PREFIX, analyze_diversity_selective_refinement,
    diversity_selective_refinement_figures, fetch_diversity_selective_refinement)

EXPERIMENT_PREFIX = DIVERSITY_V2_EXPERIMENT_PREFIX
FIXED_THRESHOLD = DIVERSITY_FIXED_REFINEMENT_THRESHOLD  # predeclared 0.03
OUTPUT = Path("diversity_selective_refinement_v2_outputs")
OUTPUT.mkdir(exist_ok=True)

store = SupabaseStore()
rollouts, steps = fetch_diversity_selective_refinement(
    store, experiment_prefix=EXPERIMENT_PREFIX)
counts = (rollouts.groupby(["member_index", "method"]).size()
          .rename("n").reset_index())
display(counts)
expected_methods = {Method.UNCERTAINTY, Method.REFINEMENT}
assert set(rollouts.method) == expected_methods
assert set(counts.n) == {1300}, "Expected 1,300 matched identities in every member/method arm"
assert len(rollouts) == 5200
print({"rollouts": len(rollouts), "step_rows": len(steps),
       "experiments": sorted(rollouts.experiment.unique()),
       "fixed_threshold": FIXED_THRESHOLD})

## 3. Build matched policies and uncertainty-horizon analyses

In [ ]:
tables = analyze_diversity_selective_refinement(
    rollouts, steps, fixed_threshold=FIXED_THRESHOLD)

overall = tables["selective_refinement_overall"]
horizons = ["first_chunk", "prefix_2_chunks", "prefix_4_chunks",
            "prefix_8_chunks", "full_episode"]
columns = ["score_name", "interpretation", "n_pairs", "best_fixed_member_sr",
           "lower_u_baseline_sr", "lower_u_refine_all_sr",
           "n_refined_fixed", "fixed_threshold_sr",
           "fixed_delta_vs_lower_u_pp", "fixed_delta_ci_low_pp",
           "fixed_delta_ci_high_pp", "fixed_F_to_S", "fixed_S_to_F"]
print("Primary 0.03 gate and uncertainty-horizon results")
display(overall[overall.score_name.isin(horizons)][columns])

print("Individual chunks (chunk 0 is deployable; later chunks are post-hoc here)")
display(overall[overall.signal_kind == "individual_chunk"][columns])

## 4. Exploratory windows and cross-validated estimate

The top-window table is selected and evaluated on the same data, so it is an optimistic discovery
tool. LOSO chooses a window on 12 suites and applies it once to the held-out suite; its pooled
delta is the more defensible estimate of whether a learned window transfers.

In [ ]:
top = tables["selective_refinement_top_windows"]
print("Top in-sample windows")
display(top[(top.score_name.isin(horizons)) & (top["rank"] <= 5)][
    ["score_name", "rank", "lower", "upper", "n_refined", "selective_sr",
     "delta_pp", "selected_F_to_S", "selected_S_to_F"]])

print("Leave-one-suite-out window selection")
display(tables["selective_refinement_loso_summary"])

print("Primary first-chunk fixed-threshold result by suite")
by_suite = tables["selective_refinement_fixed_by_suite"]
display(by_suite[by_suite.score_name == "first_chunk"][[
    "suite", "n_pairs", "lower_u_baseline_sr", "n_refined_fixed",
    "fixed_threshold_sr", "fixed_delta_vs_lower_u_pp", "fixed_F_to_S", "fixed_S_to_F"]])

## 5. Save tables and figures

In [ ]:
for name, frame in tables.items():
    frame.to_csv(OUTPUT / f"{name}.csv", index=False)
paths = diversity_selective_refinement_figures(tables, OUTPUT / "figures")
for path in paths:
    print(path.name)
    display(Image(filename=str(path)))

## 6. Concise readout

In [ ]:
overall = tables["selective_refinement_overall"].set_index("score_name")
loso = tables["selective_refinement_loso_summary"].set_index("score_name")
for score_name in ("first_chunk", "full_episode"):
    row, cv = overall.loc[score_name], loso.loc[score_name]
    print(score_name.replace("_", " "))
    print("  lower-U baseline: %.1f%%" % (100 * row.lower_u_baseline_sr))
    print("  fixed U >= %.2f: %.1f%% (%+.1f pp; F->S=%d, S->F=%d)" %
          (row.fixed_threshold, 100 * row.fixed_threshold_sr,
           row.fixed_delta_vs_lower_u_pp, row.fixed_F_to_S, row.fixed_S_to_F))
    print("  refine selected model always: %.1f%% (%+.1f pp)" %
          (100 * row.lower_u_refine_all_sr, row.refine_all_delta_vs_lower_u_pp))
    print("  LOSO window: %.1f%% (%+.1f pp, CI [%+.1f, %+.1f])" %
          (100 * cv.loso_selective_sr, cv.delta_pp,
           cv.delta_ci_low_pp, cv.delta_ci_high_pp))
print("\nInterpretation: first chunk is deployable; full episode and later chunks are post-hoc.")